# 🎙️ Train MY Voice — XTTS-v2 Fine-Tune (Google Colab, free T4 GPU)

This notebook is the **full show**: it runs a guided recording session (it tells
you exactly what to read), builds a training dataset from your recordings,
**fine-tunes the XTTS-v2 voice model on YOUR voice** (this is what closes the
gap to ~100% — your voice gets baked into the model weights, not imitated),
evaluates the result against your real voice, and then **narrates any article
you paste** — in your voice.

**Before anything else:** menu **Runtime → Change runtime type → T4 GPU → Save.**
(Do this FIRST — changing it later wipes the VM.)

| Phase | What happens | Time |
|---|---|---|
| 1. Record | You read ~13 guided blocks (~20–25 min of speech) | ~35 min of your time |
| 2. Dataset | Whisper transcribes + segments your audio | ~5–10 min |
| 3. Fine-tune | XTTS-v2 trains on your voice (T4 GPU) | ~1–2 h |
| 4. Evaluate | Fine-tuned vs. base model, similarity scores | ~5 min |
| 5. Narrate | Paste an article → WAV in your voice | minutes |

Everything is saved to your Google Drive as it happens, so a disconnected
session never loses your recordings — re-run cells 1–3 and continue from the
phase you were in (each phase cell restores what it needs from Drive).
**Keep this tab open during training** — free Colab disconnects idle tabs.

> ⚖️ **License**: XTTS-v2 weights are CPML (non-commercial). Fine for personal
> article narration; for commercial use a differently-licensed model is needed.
> 🔒 **Privacy**: your voice is a credential. The recordings and the trained
> model stay in *your* Drive — share them with no one.

In [ ]:
#@title 1 · Environment check (GPU + versions)
import sys, subprocess, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__, "| cuda:", torch.version.cuda)
gpu_ok = torch.cuda.is_available()
print("GPU    :", torch.cuda.get_device_name(0) if gpu_ok else "NONE")
if not gpu_ok:
    print("\n*** No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun. ***")
    print("(Recording + transcription also work on CPU, but training needs the GPU.)")
    print("If Colab says no GPU is available, your free quota may be used up —")
    print("record + transcribe now on CPU, and come back for training later.")
subprocess.run(["nvidia-smi", "--query-gpu=memory.total,memory.used",
                "--format=csv"], check=False)

In [ ]:
#@title 2 · Mount Google Drive (approve the popup) + project layout
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, glob, json

LANG        = "en"   # language of your recordings & articles ("it" for Italian)
DRIVE_ROOT  = "/content/drive/MyDrive/my_voice_clone"
LOCAL_ROOT  = "/content/my_voice_clone"

REC_DRIVE   = f"{DRIVE_ROOT}/recordings"       # your raw voice blocks (precious!)
DS_DRIVE    = f"{DRIVE_ROOT}/dataset"          # segmented training dataset
FT_DRIVE    = f"{DRIVE_ROOT}/finetuned"        # inference-ready fine-tuned bundle
STATE_DRIVE = f"{DRIVE_ROOT}/train_state"      # trainer checkpoint for resume
OUT_DRIVE   = f"{DRIVE_ROOT}/outputs"          # narrated articles
BASE_LOCAL  = f"{LOCAL_ROOT}/base_model"       # XTTS-v2 base files (re-downloadable)
DS_LOCAL    = f"{LOCAL_ROOT}/dataset"          # local working copy (fast disk)
RUN_LOCAL   = f"{LOCAL_ROOT}/run_out"          # trainer output (local, then Drive)

for d in (REC_DRIVE, DS_DRIVE, FT_DRIVE, STATE_DRIVE, OUT_DRIVE,
          BASE_LOCAL, DS_LOCAL, RUN_LOCAL):
    os.makedirs(d, exist_ok=True)

def _drive_status():
    recs = sorted(glob.glob(f"{REC_DRIVE}/*.wav"))
    meta = os.path.exists(f"{DS_DRIVE}/metadata_train.csv")
    ft   = os.path.exists(f"{FT_DRIVE}/model.pth")
    st   = os.path.exists(f"{STATE_DRIVE}/best_model.pth")
    print(f"recordings on Drive : {len(recs)} blocks")
    print(f"dataset built       : {meta}")
    print(f"fine-tuned model    : {ft}")
    print(f"resumable train state: {st}")
_drive_status()

In [ ]:
#@title 3 · Install (pinned — these exact pins avoid known breakage)
# coqui-tts 0.27.5: torch/numpy are NOT touched (they're optional extras).
# transformers MUST stay <5 (5.x removed a symbol coqui imports -> ImportError).
%pip install -q "coqui-tts==0.27.5" "transformers>=4.57,<5" "faster-whisper>=1.0"

import os, sys, subprocess, torch
os.environ["COQUI_TOS_AGREED"] = "1"   # accept Coqui CPML (non-commercial) terms

# torchcodec is REQUIRED by coqui-tts on torch>=2.9 and must match torch:
mm = tuple(map(int, torch.__version__.split("+")[0].split(".")[:2]))
pin = {(2, 9): "torchcodec==0.9.1", (2, 10): "torchcodec==0.10.0"}.get(
    mm, "torchcodec>=0.13" if mm >= (2, 11) else None)
if pin:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)

# --- probe: can the audio backend actually decode on this VM? ----------------
def _probe():
    subprocess.run("ffmpeg -hide_banner -loglevel error -y -f lavfi -i "
                   "sine=frequency=440:duration=1 -ar 22050 /content/_probe.wav",
                   shell=True, check=True)
    if mm >= (2, 9):                    # torchcodec only required on torch>=2.9
        from torchcodec.decoders import AudioDecoder
        AudioDecoder("/content/_probe.wav")
    import TTS
    print("✓ coqui-tts", TTS.__version__, "· audio backend OK · torch", torch.__version__)

try:
    _probe()
except Exception as e:                      # known Colab case: FFmpeg 4 too old
    print("probe failed:", e, "\n→ FIX 1: installing newer FFmpeg shared libs...")
    subprocess.run("add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg7 "
                   "&& apt-get -qq update && apt-get -y -qq install ffmpeg",
                   shell=True, check=False)
    try:
        _probe()
    except Exception as e2:
        # FIX 2 (automatic): drop to torch 2.8 — it needs no torchcodec at all.
        # This takes ~5 min, then the runtime restarts itself.
        print("still failing:", e2)
        print("\n→ FIX 2: switching to torch 2.8 (no torchcodec needed). ~5 min...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0",
                        "--index-url", "https://download.pytorch.org/whl/cu126"],
                       check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                        "torchcodec"], check=False)
        print("\n" + "=" * 62)
        print("FIXED. The runtime will now RESTART ITSELF (this is normal).")
        print("After it restarts: click  Runtime → Run all  once more.")
        print("=" * 62)
        import time as _t; _t.sleep(5)
        os.kill(os.getpid(), 9)             # standard Colab self-restart

## Phase 1 — Guided recording session 🎤

I direct, you speak. Each block below shows a text and records you through the
browser microphone (Chrome will ask for permission on the first one — allow it).

**How to record well** (this matters more than anything):
- Quiet room, no fan/AC/traffic; soft furnishings kill echo.
- Keep the same distance to the mic (~15–20 cm) for ALL blocks.
- Read **naturally, as if to one listener** — not like a robot, not like a stage.
- Match each block's style (calm / emphatic / conversational — it's labeled).
- Fluffed a line? Just keep going — small stumbles are fine. Botched the whole
  block? `rerecord("n2")` re-does it.
- Recording stops automatically after the allotted time. Finished early? Just
  stay silent — trailing silence is trimmed automatically.

In [ ]:
#@title 4 · The recording script (what you will read)
# Guided recording session — the blocks Giorgio reads, in order.
# Each block: style, mode ('read' = exact text, transcript known;
# 'free' = natural speech, transcribed by whisper), target seconds.
# Total ≈ 22–28 minutes of speech — squarely in the fine-tune sweet spot.

BLOCKS = [
    # ---------------- NEUTRAL NARRATION (the default article voice) ----------
    dict(id="n1", style="neutral", mode="read", target_s=75, title="Calm narration — security",
         text=("Smart contract security is not about finding one bug. It is about understanding how a system "
               "behaves under pressure — how its pieces interact when someone is actively trying to break them. "
               "We read code the way an attacker would: patiently, looking for the assumption that everyone else "
               "took for granted. Most exploits are not clever. They are simple mistakes that nobody checked. "
               "A missing access control. A price that can be moved within a single transaction. A callback that "
               "arrives earlier than the developer expected. The difference between a safe protocol and a headline "
               "is usually one overlooked line of code.")),
    dict(id="n2", style="neutral", mode="read", target_s=70, title="Calm narration — storytelling",
         text=("The harbor was quiet that morning. A thin fog rolled over the water, softening the outlines of the "
               "boats, and the only sound was the slow creak of rope against wood. She walked along the pier with "
               "her coffee, watching the gulls argue over something near the lighthouse. Every town has an hour "
               "like this — before the shops open, before the traffic starts — when the whole place seems to "
               "breathe in and hold it. By eight it would be gone. But for now, the morning belonged to her.")),
    dict(id="n3", style="neutral", mode="read", target_s=70, title="Numbers, dates, spelling",
         text=("The audit began on March third, twenty twenty-six, and lasted fourteen working days. We reviewed "
               "eleven contracts totalling roughly six thousand four hundred lines of code. The team found three "
               "critical issues, seven medium, and nineteen informational findings. Version two point five point "
               "one fixed all of them. The total value secured exceeded two hundred and thirty million dollars. "
               "You can reach us at contact at dedaub dot com, or visit dedaub dot com slash audits. The report "
               "number is A-forty-seven, dated April tenth.")),
    dict(id="n4", style="neutral", mode="read", target_s=70, title="Explaining a concept simply",
         text=("Think of a flash loan as borrowing money that must be returned before you leave the room. Within a "
               "single transaction, you can borrow millions with no collateral — as long as, by the end of that "
               "same transaction, everything is paid back. Attackers love this because it gives them enormous "
               "temporary leverage. They borrow, they push a price, they profit from the distortion, and they "
               "repay — all in one atomic step. If any part fails, the whole thing simply never happened.")),
    dict(id="n5", style="neutral", mode="read", target_s=65, title="Phonetic coverage",
         text=("The quick brown fox jumps over the lazy dog while zebras vex the judge. She sells sea shells by "
               "the seashore, and the shore is never truly silent. Bright blue balloons floated above the crowded "
               "market square. Thick fog rolled in from the harbor as the last ferry departed. Please pour the "
               "water into the tall glass without spilling a single drop. Autumn leaves drifted down, red and "
               "gold, onto the damp cobblestones, and distant thunder rumbled while the children counted seconds.")),

    # ---------------- EMPHATIC (headline / persuasive voice) -----------------
    dict(id="e1", style="emphatic", mode="read", target_s=60, title="Emphatic — push energy on key words",
         text=("Stop. Read that number again, because it matters. This is not a small improvement — it is a "
               "fundamental shift in how the system works. We did not just fix the bug; we eliminated the entire "
               "class of bugs. Never — and I mean never — ship that to production untested. You have one chance "
               "to get this right, so make it count. The difference between safe and exploited is one missing "
               "check. If you remember one thing from this article, remember this.")),
    dict(id="e2", style="emphatic", mode="read", target_s=60, title="Emphatic — announcement",
         text=("Today we are announcing something we have worked on for two years. The results were not good — "
               "they were extraordinary. Every single test passed. Every benchmark improved. And here is the part "
               "nobody tells you: this changes everything about how audits are done. Pay attention, because this "
               "is where most people get it wrong. That is not a feature. That is a completely new way of "
               "thinking about security.")),

    # ---------------- CONVERSATIONAL (questions, lighter) --------------------
    dict(id="c1", style="conversational", mode="read", target_s=60, title="Conversational — questions",
         text=("So, what actually happens when the oracle returns a stale price? Ever wonder why some audits catch "
               "this and others just don't? Right? It seems obvious once someone finally points it out. Now, you "
               "might be thinking — isn't that a bit paranoid? But here's a question worth sitting with: who "
               "verifies the verifier? Interesting, isn't it, how the simplest bugs cause the biggest losses? "
               "Okay, so where do we go from here? Let's walk through it together.")),
    dict(id="c2", style="conversational", mode="read", target_s=55, title="Conversational — aside",
         text=("Look, I get it — nobody wakes up excited about reading audit reports. And honestly? Most of them "
               "are written like nobody is supposed to read them at all. But here's the thing: the good ones tell "
               "a story. They tell you what almost went wrong, and how close it came. Have you ever shipped code "
               "you were absolutely sure was correct? Yeah. That's the feeling we're talking about.")),

    # ---------------- DOMAIN VOCABULARY --------------------------------------
    dict(id="d1", style="neutral", mode="read", target_s=65, title="Your work vocabulary",
         text=("Dedaub audits smart contracts, monitors protocols, and builds static analysis tools. Reentrancy, "
               "integer overflow, and access-control flaws remain the classics. The oracle was manipulated through "
               "a flash-loan-funded price swing. We reviewed the ERC-twenty token, the automated market maker, and "
               "the upgradeable proxy pattern. Chainlink, EigenLayer, and the Ethereum Foundation trust rigorous "
               "review. Governance, delegatecall, and storage collisions deserve real scrutiny. Real-time "
               "monitoring flags anomalies the moment a transaction lands on chain.")),

    # ---------------- FREE NATURAL SPEECH (whisper-transcribed) --------------
    dict(id="f1", style="neutral", mode="free", target_s=90, title="Free speech — describe your work",
         text=("SPEAK FREELY for about 90 seconds: describe what you do at work — your typical day, what Dedaub "
               "does, what you enjoy about it. Don't read anything. Talk exactly as if explaining to a friend. "
               "Pauses, restarts and 'ehm's are fine — that's your real voice.")),
    dict(id="f2", style="conversational", mode="free", target_s=75, title="Free speech — tell a story",
         text=("SPEAK FREELY for about 75 seconds: tell a short story — a trip you took, a memorable meeting, "
               "something funny that happened recently. Natural conversational tone, like a voice message to a "
               "friend.")),
]

# ≈ sum of target_s ≈ 14.5 min minimum; users naturally overshoot ~1.5x → ~20-25 min

print(f"{len(BLOCKS)} blocks · ~" +
      f"{sum(b['target_s'] for b in BLOCKS)//60} min minimum speech")
for b in BLOCKS:
    print(f"  {b['id']:>3} [{b['style']:<14}] {b['title']}")

In [ ]:
#@title 5 · Recorder (browser microphone → clean WAV on Drive)
import wave as wave_mod
import numpy as np
from base64 import b64decode
from google.colab import output as colab_output
from IPython.display import display, HTML, Javascript, Audio

RECORD_JS = """
const sleep = t => new Promise(r => setTimeout(r, t));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.onstop = async () => {
    stream.getTracks().forEach(t => t.stop());
    resolve(await b2text(new Blob(chunks)));
  };
  recorder.start();
  await sleep(time);
  recorder.stop();
});
"""

def _trim_silence(x, sr, thresh=0.01, pad_s=0.25):
    idx = np.where(np.abs(x) > thresh)[0]
    if len(idx) == 0:
        return x
    pad = int(pad_s * sr)
    return x[max(0, idx[0] - pad): min(len(x), idx[-1] + pad)]

def _wav_stats(path):
    with wave_mod.open(path, "rb") as w:
        sr = w.getframerate()
        x = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16
                          ).astype(np.float32) / 32768.0
    return x, sr

def record_block(block_id, extra_s=0):
    b = next(bb for bb in BLOCKS if bb["id"] == block_id)
    secs = int(b["target_s"] * 1.5) + int(extra_s)
    display(HTML(
        f"<div style='border:2px solid #d97706;border-radius:10px;padding:16px;"
        f"max-width:820px;font-size:17px;line-height:1.6'>"
        f"<b>[{b['id']}] {b['title']}</b> — style: <b>{b['style']}</b> · "
        f"recording for <b>{secs}s</b> (finish early? just stay silent)<hr>"
        f"{b['text']}</div>"))
    display(Javascript(RECORD_JS))
    print(f"● RECORDING {secs}s — read now...")
    try:
        data_url = colab_output.eval_js("record(%d)" % (secs * 1000))
    except Exception:
        print("✗ Could not access the microphone. Click the mic/camera icon in "
              "the browser address bar, ALLOW microphone for this site, then "
              f"rerun: record_block('{block_id}')")
        return None
    webm = b64decode(data_url.split(",")[1])
    webm_path = f"/content/_{block_id}.webm"
    with open(webm_path, "wb") as f:
        f.write(webm)
    wav_path = f"{REC_DRIVE}/{block_id}.wav"
    tmp = f"/content/_{block_id}.wav"
    # MediaRecorder gives webm/opus 48k — convert to the XTTS rate: 22050 mono
    subprocess.run(f"ffmpeg -y -hide_banner -loglevel error -i {webm_path} "
                   f"-ac 1 -ar 22050 {tmp}", shell=True, check=True)
    x, sr = _wav_stats(tmp)
    x = _trim_silence(x, sr)
    dur, peak = len(x) / sr, float(np.max(np.abs(x))) if len(x) else 0.0
    with wave_mod.open(wav_path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((np.clip(x, -1, 1) * 32767).astype(np.int16).tobytes())
    verdict = []
    if dur < b["target_s"] * 0.5: verdict.append("SHORT — consider rerecord")
    if peak > 0.98: verdict.append("CLIPPING — move back from the mic, rerecord")
    if peak < 0.05: verdict.append("VERY QUIET — check mic, rerecord")
    print(f"✓ saved {block_id}.wav  ({dur:.1f}s, peak {peak:.2f}) "
          + ("· " + "; ".join(verdict) if verdict else "· looks good"))
    display(Audio(wav_path))
    return wav_path

def rerecord(block_id, extra_s=15):
    return record_block(block_id, extra_s=extra_s)

def record_all_remaining():
    for b in BLOCKS:
        if os.path.exists(f"{REC_DRIVE}/{b['id']}.wav"):
            print(f"— {b['id']} already recorded, skipping")
            continue
        record_block(b["id"])
        input("Press Enter for the next block (or interrupt to stop)...")

def review_recordings():
    total = 0.0
    for b in BLOCKS:
        p = f"{REC_DRIVE}/{b['id']}.wav"
        if os.path.exists(p):
            x, sr = _wav_stats(p); total += len(x) / sr
            print(f"{b['id']:>3} {len(x)/sr:6.1f}s  {b['title']}")
            display(Audio(p))
        else:
            print(f"{b['id']:>3}   ---   NOT RECORDED — {b['title']}")
    print(f"\nTOTAL: {total/60:.1f} min "
          f"({'enough for fine-tuning ✓' if total >= 600 else 'aim for 15+ min'})")

print("Ready. Run:  record_all_remaining()   — or record_block('n1') one by one.")

In [ ]:
#@title 6 · ▶ RECORD — run this and follow the prompts
record_all_remaining()
print("\nAll blocks done! Run review_recordings() to listen / rerecord(id) to redo.")

In [ ]:
#@title 7 · (Alternative) upload recordings made elsewhere
# Recorded on your phone / Audacity instead? Upload files here. Name them by
# block id (n1.wav, e2.m4a ...) to slot them in; any other name is added as
# extra free-speech material (whisper transcribes everything anyway).
from google.colab import files
up = files.upload()
known = {b["id"] for b in BLOCKS}
n = len(glob.glob(f"{REC_DRIVE}/up*.wav"))     # don't overwrite earlier uploads
for name, _ in up.items():
    stem = os.path.splitext(name)[0].lower()
    tgt = stem if stem in known else f"up{n}"; n += 1
    subprocess.run(f"ffmpeg -y -hide_banner -loglevel error -i '/content/{name}' "
                   f"-ac 1 -ar 22050 {REC_DRIVE}/{tgt}.wav", shell=True, check=True)
    print(f"✓ {name} -> {tgt}.wav")

## Phase 2 — Build the training dataset 🧩

Whisper (large-v3) listens to every recording, writes down **what you actually
said** with word-level timestamps, and slices the audio into sentence-sized
clips (0.6–11 s) — the format XTTS training expects. Output: `wavs/*.wav` +
`metadata_train.csv` / `metadata_eval.csv`, saved to Drive.

In [ ]:
#@title 8 · Transcribe + segment (faster-whisper large-v3)
import csv, gc, glob, random
import numpy as np
import torch  # imported BEFORE WhisperModel: preloads cuDNN libs (known fix)
import wave as wave_mod
from faster_whisper import WhisperModel

SR = 22050
recs = sorted(glob.glob(f"{REC_DRIVE}/*.wav"))
assert recs, "No recordings found on Drive — do Phase 1 first."
print(f"{len(recs)} recordings to process")

device = "cuda" if torch.cuda.is_available() else "cpu"
whisper = WhisperModel("large-v3", device=device,
                       compute_type="float16" if device == "cuda" else "int8")

os.makedirs(f"{DS_LOCAL}/wavs", exist_ok=True)
rows, buffer_s, MAX_S, MIN_S = [], 0.2, 11.0, 0.6

def _read_wav(path):
    with wave_mod.open(path, "rb") as w:
        assert w.getframerate() == SR, f"{path}: expected {SR} Hz"
        return np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16
                             ).astype(np.float32) / 32768.0

def _write_wav(path, x):
    with wave_mod.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        w.writeframes((np.clip(x, -1, 1) * 32767).astype(np.int16).tobytes())

def _flush(words, audio, src, k):
    """Write one sentence chunk (splitting further if > MAX_S)."""
    if not words:
        return k
    text = " ".join(w.word.strip() for w in words).strip()
    start, end = words[0].start, words[-1].end
    if end - start > MAX_S:
        if len(words) == 1:                       # degenerate timestamp: hard cap
            end = start + MAX_S
        else:                                     # greedy split on word gaps
            mid = len(words) // 2
            k = _flush(words[:mid], audio, src, k)
            return _flush(words[mid:], audio, src, k)
    a = max(0, int((start - buffer_s) * SR))
    b = min(len(audio), int((end + buffer_s) * SR))
    if (b - a) / SR < MIN_S or len(text) < 3:
        return k
    name = f"wavs/{src}_{k:04d}.wav"
    _write_wav(f"{DS_LOCAL}/{name}", audio[a:b])
    rows.append((name, text))
    return k + 1

for path in recs:
    src = os.path.splitext(os.path.basename(path))[0]
    audio = _read_wav(path)
    segments, _info = whisper.transcribe(path, language=LANG,
                                         word_timestamps=True, beam_size=5)
    words = [w for seg in segments for w in (seg.words or [])]
    sent, k = [], 0
    for w in words:
        sent.append(w)
        if w.word.strip().endswith((".", "!", "?")):
            k = _flush(sent, audio, src, k); sent = []
    k = _flush(sent, audio, src, k)
    print(f"  {src}: {k} clips")

del whisper; gc.collect(); torch.cuda.empty_cache()   # free VRAM for training

assert len(rows) >= 30, f"Only {len(rows)} clips — record more material."
random.seed(7); random.shuffle(rows)
n_eval = max(2, int(len(rows) * 0.15))
def _write_meta(fname, subset):                        # 'coqui' formatter: header!
    with open(f"{DS_LOCAL}/{fname}", "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f, delimiter="|")
        wr.writerow(["audio_file", "text", "speaker_name"])
        for name, text in subset:
            wr.writerow([name, text, "me"])
_write_meta("metadata_eval.csv", rows[:n_eval])
_write_meta("metadata_train.csv", rows[n_eval:])
total_s = sum(os.path.getsize(p) for p in glob.glob(f"{DS_LOCAL}/wavs/*.wav")) / (SR * 2)
print(f"\n✓ dataset: {len(rows)} clips (~{total_s/60:.1f} min) "
      f"→ {len(rows)-n_eval} train / {n_eval} eval")
subprocess.run(f"cp -r {DS_LOCAL}/. {DS_DRIVE}/", shell=True, check=True)
print("✓ copied to Drive")

## Phase 3 — Fine-tune XTTS-v2 on your voice 🔥

This is the step no zero-shot tool does: gradient descent on **your** clips.
Uses the official Coqui recipe values (AdamW, lr 5e-6) with the XTTS-v2
overrides. ~1–2 h on the free T4 for 10 epochs.

**⚠️ Keep this browser tab OPEN and click around occasionally during training**
— free Colab disconnects idle tabs (~90 min) even mid-run. If it does
disconnect: re-run cells 1–3, then this cell — it restores the dataset and
base files from Drive and **warm-starts from the newest saved checkpoint**
(the epoch counter restarts, but the learned weights carry over — you lose
minutes, not hours).

Watch `avg_loss_mel_ce` on the eval runs: it should fall then flatten. If it
starts *rising* while train loss falls, you're overfitting — stop; the best
checkpoint is already saved.

In [ ]:
#@title 9 · Download base model files (cached on Drive)
from TTS.utils.manage import ModelManager

HF = "https://huggingface.co/coqui/XTTS-v2/resolve/main/"
FILES = ["model.pth", "config.json", "vocab.json", "dvae.pth", "mel_stats.pth"]
BASE_DRIVE = f"{DRIVE_ROOT}/base_model"
os.makedirs(BASE_DRIVE, exist_ok=True)
need = [f for f in FILES if not os.path.exists(f"{BASE_DRIVE}/{f}")]
if need:
    ModelManager._download_model_files([HF + f for f in need], BASE_DRIVE,
                                       progress_bar=True)
subprocess.run(f"cp -n {BASE_DRIVE}/* {BASE_LOCAL}/", shell=True, check=True)
for f in FILES:
    sz = os.path.getsize(f"{BASE_LOCAL}/{f}") / 1e6
    print(f"  {f:14s} {sz:8.1f} MB")

In [ ]:
#@title 10 · TRAIN (resumable — safe to re-run after a disconnect)
import sys, threading, time, shutil, glob
sys.argv = [""]      # Trainer parses sys.argv; strip the Jupyter kernel's -f arg

from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import (GPTArgs, GPTTrainer,
                                                     GPTTrainerConfig)
from TTS.tts.models.xtts import XttsAudioConfig

NUM_EPOCHS  = 10   # 6-15 sensible for ~20 min of audio; watch eval loss
BATCH_SIZE  = 4    # OOM? -> 2 (and GRAD_ACUMM -> 2)
GRAD_ACUMM  = 1

# --- self-healing after a disconnect: restore everything from Drive ----------
if not os.path.exists(f"{DS_LOCAL}/metadata_train.csv"):
    subprocess.run(f"cp -r {DS_DRIVE}/. {DS_LOCAL}/", shell=True, check=True)
BASE_DRIVE = f"{DRIVE_ROOT}/base_model"
for f in ("model.pth", "config.json", "vocab.json", "dvae.pth", "mel_stats.pth"):
    if not os.path.exists(f"{BASE_LOCAL}/{f}"):
        src = f"{BASE_DRIVE}/{f}"
        assert os.path.exists(src), f"{f} missing — run cell 9 (base download) first"
        shutil.copy(src, f"{BASE_LOCAL}/{f}")

model_args = GPTArgs(
    max_conditioning_length=132300, min_conditioning_length=66150,
    debug_loading_failures=True,          # print (don't silently drop) bad clips
    max_wav_length=255995, max_text_length=200,
    mel_norm_file=f"{BASE_LOCAL}/mel_stats.pth",
    dvae_checkpoint=f"{BASE_LOCAL}/dvae.pth",
    xtts_checkpoint=f"{BASE_LOCAL}/model.pth",
    tokenizer_file=f"{BASE_LOCAL}/vocab.json",
    gpt_num_audio_tokens=1026, gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True, gpt_use_perceiver_resampler=True,
)
config = GPTTrainerConfig(
    run_name="my_voice", output_path=RUN_LOCAL, model_args=model_args,
    audio=XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050,
                          output_sample_rate=24000),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE,
    batch_group_size=48, num_loader_workers=2, eval_split_max_size=256,
    print_step=25, plot_step=100, save_step=250, save_n_checkpoints=1,
    save_checkpoints=True, print_eval=False,
    optimizer="AdamW", optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-06, lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [900000, 2700000, 5400000],
                         "gamma": 0.5, "last_epoch": -1},
    test_sentences=[],
)
dataset_cfg = BaseDatasetConfig(
    formatter="coqui", dataset_name="mine", path=DS_LOCAL,
    meta_file_train="metadata_train.csv", meta_file_val="metadata_eval.csv",
    language=LANG,
)
train_samples, eval_samples = load_tts_samples([dataset_cfg], eval_split=True)
print(f"samples: {len(train_samples)} train / {len(eval_samples)} eval")

model = GPTTrainer.init_from_config(config)

# --- resume strategy ---------------------------------------------------------
# STATE_DRIVE/run mirrors the trainer run folder (best_model.pth + config.json,
# ~5.4 GB — the ONLY full checkpoint kept on Drive). On a fresh VM we copy it
# back and use continue_path => TRUE resume: optimizer/scheduler state, epoch
# counter and best-loss guard all restored (plain restore_path would restart
# epochs and let a worse first eval overwrite the best checkpoint).
resume_dir = f"{STATE_DRIVE}/run"
os.makedirs(resume_dir, exist_ok=True)

def _make_trainer():
    if os.path.exists(f"{resume_dir}/best_model.pth"):
        local_resume = f"{RUN_LOCAL}/resume_run"
        subprocess.run(f"rm -rf {local_resume} && cp -r {resume_dir} {local_resume}",
                       shell=True, check=True)
        try:
            t = Trainer(TrainerArgs(continue_path=local_resume,
                                    grad_accum_steps=GRAD_ACUMM),
                        config, output_path=RUN_LOCAL, model=model,
                        train_samples=train_samples, eval_samples=eval_samples,
                        parse_command_line_args=False)
            print("↻ TRUE resume from Drive checkpoint (optimizer + epoch restored)")
            return t
        except Exception as e:
            print("continue_path resume failed, warm-starting from weights:", e)
            return Trainer(TrainerArgs(restore_path=f"{resume_dir}/best_model.pth",
                                       grad_accum_steps=GRAD_ACUMM),
                           config, output_path=RUN_LOCAL, model=model,
                           train_samples=train_samples, eval_samples=eval_samples,
                           parse_command_line_args=False)
    return Trainer(TrainerArgs(grad_accum_steps=GRAD_ACUMM), config,
                   output_path=RUN_LOCAL, model=model,
                   train_samples=train_samples, eval_samples=eval_samples,
                   parse_command_line_args=False)

trainer = _make_trainer()

_stop_mirror = threading.Event()

def _safe_mirror(src, dst, min_age_s=30):
    """Copy only a *stable* src (not mid-write), atomically, with a Drive-side
    size check. The trainer rewrites best_model.pth non-atomically over several
    seconds — copying during that window would corrupt the resume file."""
    if not os.path.exists(src):
        return
    s1 = os.stat(src); time.sleep(4); s2 = os.stat(src)
    if (s1.st_mtime, s1.st_size) != (s2.st_mtime, s2.st_size):
        return                                   # being written right now
    if time.time() - s2.st_mtime < min_age_s:
        return                                   # too fresh — wait a cycle
    if os.path.exists(dst) and os.path.getmtime(dst) >= s2.st_mtime:
        return                                   # unchanged since last mirror
    tmp = dst + ".tmp"
    shutil.copy(src, tmp)
    if os.path.getsize(tmp) != s2.st_size:       # FUSE write-back sanity check
        os.remove(tmp); raise IOError("Drive copy size mismatch")
    os.replace(tmp, dst)
    print(f"  [mirror] checkpoint → Drive ({s2.st_size/1e9:.1f} GB)")

def _mirror_run_folder(min_age_s=30):
    _safe_mirror(os.path.join(trainer.output_path, "best_model.pth"),
                 f"{resume_dir}/best_model.pth", min_age_s=min_age_s)
    cfg_json = os.path.join(trainer.output_path, "config.json")
    if os.path.exists(cfg_json):
        shutil.copy(cfg_json, f"{resume_dir}/config.json")

def _mirror_loop():
    while not _stop_mirror.wait(600):
        try:
            _mirror_run_folder()
        except Exception as e:
            print("  [mirror] skipped:", e)
_mirror_thread = threading.Thread(target=_mirror_loop, daemon=True)
_mirror_thread.start()

try:
    trainer.fit()
finally:
    _stop_mirror.set()                           # stop the mirror thread
    _mirror_thread.join(timeout=60)              # no overlap with final copies

best = os.path.join(trainer.output_path, "best_model.pth")
assert os.path.exists(best), "training produced no best_model.pth"
_mirror_run_folder(min_age_s=0)         # final full-checkpoint mirror (resume)

# free the ~7 GB of GPU memory the trainer holds, BEFORE eval loads models
import gc
del trainer, model
gc.collect(); torch.cuda.empty_cache()

# --- inference-ready SLIM bundle (weights only, ~2 GB instead of ~5.5 GB) ----
# Drive quota math (free tier = 15 GB): base cache ~2 GB + full resume
# checkpoint ~5.5 GB + slim bundle ~2 GB ≈ 9.5 GB. Shipping the full trainer
# checkpoint here too would blow the quota — so we strip optimizer/DVAE state.
ckpt = torch.load(best, map_location="cpu", weights_only=True)
slim_model = {k: v for k, v in ckpt["model"].items()
              if not k.startswith(("dvae.", "torch_mel_spectrogram"))}
slim_local = f"{RUN_LOCAL}/slim_model.pth"       # write locally, copy atomically
torch.save({"model": slim_model}, slim_local)
_safe_mirror(slim_local, f"{FT_DRIVE}/model.pth", min_age_s=0)
shutil.copy(f"{BASE_LOCAL}/config.json", f"{FT_DRIVE}/config.json")
shutil.copy(f"{BASE_LOCAL}/vocab.json", f"{FT_DRIVE}/vocab.json")
del ckpt, slim_model; gc.collect()
print("\n✓ fine-tuned bundle saved to Drive:", FT_DRIVE,
      f"({os.path.getsize(FT_DRIVE + '/model.pth')/1e9:.1f} GB)")

## Phase 4 — Evaluate: does it sound like YOU? 📊

Synthesizes test lines with the **base** model and with **your fine-tuned**
model, then scores each against your real recordings using the model's own
speaker encoder (cosine similarity). Listen to both — the numbers guide, your
ears decide.

In [ ]:
#@title 11 · Evaluate fine-tuned vs base
import gc, glob
import numpy as np
import torch, torchaudio
from IPython.display import Audio, display
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

TEST_LINES = [
    "Here is a test of my cloned voice reading a completely new sentence.",
    "Security is a process, not a product, and it never really ends.",
    "Tomorrow morning I will publish the next article on our research blog.",
]

def _restore_from_drive():
    """Self-heal on a fresh VM: pull dataset + base files back from Drive."""
    if not glob.glob(f"{DS_LOCAL}/wavs/*.wav"):
        subprocess.run(f"cp -r {DS_DRIVE}/. {DS_LOCAL}/", shell=True, check=True)
    for f in ("model.pth", "config.json", "vocab.json"):
        if not os.path.exists(f"{BASE_LOCAL}/{f}"):
            shutil.copy(f"{DRIVE_ROOT}/base_model/{f}", f"{BASE_LOCAL}/{f}")

_restore_from_drive()
REFS = sorted(glob.glob(f"{DS_LOCAL}/wavs/*.wav"),
              key=os.path.getsize, reverse=True)[:4]
assert REFS, "no dataset on Drive either — run Phases 1-2 first"

def load_xtts(ckpt, cfg_path, vocab):
    cfg = XttsConfig(); cfg.load_json(cfg_path)
    m = Xtts.init_from_config(cfg)
    m.load_checkpoint(cfg, checkpoint_path=ckpt, vocab_path=vocab,
                      use_deepspeed=False)
    m.cuda(); return m, cfg

def spk_embed(model, wav_path):
    _, emb = model.get_conditioning_latents(audio_path=[wav_path])
    return emb.flatten().cpu().numpy()

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

results = {}
for tag, ckpt in [("base", f"{BASE_LOCAL}/model.pth"),
                  ("fine-tuned", f"{FT_DRIVE}/model.pth")]:
    model, cfg = load_xtts(ckpt, f"{BASE_LOCAL}/config.json",
                           f"{BASE_LOCAL}/vocab.json")
    gpt_cond, spk = model.get_conditioning_latents(audio_path=REFS)
    ref_emb = spk.flatten().cpu().numpy()
    sims = []
    for i, line in enumerate(TEST_LINES):
        out = model.inference(text=line, language=LANG,
                              gpt_cond_latent=gpt_cond, speaker_embedding=spk,
                              temperature=0.7, repetition_penalty=10.0)
        p = f"/content/eval_{tag}_{i}.wav"
        torchaudio.save(p, torch.tensor(out["wav"]).unsqueeze(0), 24000)
        sims.append(cosine(spk_embed(model, p), ref_emb))
        print(f"[{tag}] {sims[-1]:.3f}  «{line[:50]}...»"); display(Audio(p))
    results[tag] = float(np.mean(sims))
    del model; gc.collect(); torch.cuda.empty_cache()

print(f"\nmean similarity to your voice — base: {results['base']:.3f} "
      f"| fine-tuned: {results['fine-tuned']:.3f}")
print("(fine-tuned should be higher; if not, train more epochs or add data)")

## Phase 5 — Narrate your article 📰 → 🎧

Paste your article, pick a style, run. Long texts are split into sentences;
each sentence is generated **best-of-N** (multiple takes, auto-picking the one
closest to your voice — the refinement loop), then stitched with natural
pauses. The WAV lands in Drive `outputs/` and plays right here.

In [ ]:
#@title 12 · ▶ NARRATE (paste your article, run)
# Self-contained: works right after a reconnect without running the eval cell.
import re, gc, glob
import numpy as np
import torch, torchaudio
from IPython.display import Audio, display
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

ARTICLE = """
Paste your article text here. Every sentence will be read in your voice.
"""

STYLE = "neutral"        #@param ["neutral", "emphatic", "conversational"]
BEST_OF = 2              # takes per sentence; 3 = better but slower
PRESETS = {"neutral":        dict(temperature=0.65, speed=1.0),
           "emphatic":       dict(temperature=0.85, speed=1.04),
           "conversational": dict(temperature=0.75, speed=0.98)}

# --- helpers defined here too, so this cell never depends on the eval cell ---
if not glob.glob(f"{DS_LOCAL}/wavs/*.wav"):
    subprocess.run(f"cp -r {DS_DRIVE}/. {DS_LOCAL}/", shell=True, check=True)
for _f in ("config.json", "vocab.json"):
    if not os.path.exists(f"{BASE_LOCAL}/{_f}"):
        shutil.copy(f"{DRIVE_ROOT}/base_model/{_f}", f"{BASE_LOCAL}/{_f}")
REFS = sorted(glob.glob(f"{DS_LOCAL}/wavs/*.wav"),
              key=os.path.getsize, reverse=True)[:4]
assert REFS, "no dataset found — run Phases 1-2 first"
assert os.path.exists(f"{FT_DRIVE}/model.pth"), "no fine-tuned model — run Phase 3"

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def spk_embed(m, wav_path):
    _, emb = m.get_conditioning_latents(audio_path=[wav_path])
    return emb.flatten().cpu().numpy()

if "_narrate_model" not in globals():         # load once, reuse across reruns
    _cfg = XttsConfig(); _cfg.load_json(f"{BASE_LOCAL}/config.json")
    _narrate_model = Xtts.init_from_config(_cfg)
    _narrate_model.load_checkpoint(_cfg, checkpoint_path=f"{FT_DRIVE}/model.pth",
                                   vocab_path=f"{BASE_LOCAL}/vocab.json",
                                   use_deepspeed=False)
    _narrate_model.cuda()
model = _narrate_model
gpt_cond, spk = model.get_conditioning_latents(audio_path=REFS)
ref_emb = spk.flatten().cpu().numpy()

sentences = [s.strip() for s in
             re.split(r"(?<=[.!?])\s+", ARTICLE.strip()) if s.strip()]
print(f"{len(sentences)} sentences · style={STYLE} · best-of-{BEST_OF}\n")

pieces, gap = [], np.zeros(int(0.25 * 24000), dtype=np.float32)
for i, sent in enumerate(sentences):
    takes = []
    for n in range(BEST_OF):
        p = PRESETS[STYLE]
        out = model.inference(text=sent, language=LANG,
                              gpt_cond_latent=gpt_cond, speaker_embedding=spk,
                              temperature=p["temperature"] + 0.05 * n,
                              speed=p["speed"], repetition_penalty=10.0,
                              enable_text_splitting=True)
        wav = np.asarray(out["wav"], dtype=np.float32)
        tmp = f"/content/_take.wav"
        torchaudio.save(tmp, torch.tensor(wav).unsqueeze(0), 24000)
        takes.append((cosine(spk_embed(model, tmp), ref_emb), wav))
    takes.sort(key=lambda t: -t[0])
    pieces += [takes[0][1], gap]
    print(f"  {i+1:3d}/{len(sentences)}  sim={takes[0][0]:.3f}  «{sent[:60]}»")

full = np.concatenate(pieces) if pieces else gap
out_path = f"{OUT_DRIVE}/narration_{STYLE}_{len(sentences)}sent.wav"
torchaudio.save(out_path, torch.tensor(full).unsqueeze(0), 24000)
print(f"\n✓ {len(full)/24000:.0f}s of audio → {out_path}")
display(Audio(out_path))

## Export & wire into Claude Desktop 🔌

Your fine-tuned voice bundle is in Drive: `my_voice_clone/finetuned/`
(`model.pth` + `config.json` + `vocab.json`).

To use it on your own machine with the voice-orchestrator MCP server
(so Claude Desktop speaks with the trained voice):
1. Download the `finetuned` folder from Drive.
2. In `voice-orchestrator/config.yaml` set
   `backend.finetuned_dir: /path/to/finetuned`.
3. Restart Claude Desktop — `speak` now uses YOUR trained voice.

Keep the bundle private — it can say anything in your voice. The XTTS-v2
license (CPML) allows personal, non-commercial use.

In [ ]:
#@title 13 · Zip the bundle for download (optional)
# Note: the zip (~2 GB) is built on LOCAL disk and downloaded directly — it is
# NOT stored on Drive, to stay inside the free 15 GB quota.
from google.colab import files as colab_files
zip_path = "/content/my_voice_finetuned.zip"
subprocess.run(f"cd {FT_DRIVE} && zip -q -r {zip_path} .", shell=True, check=True)
print(f"{os.path.getsize(zip_path)/1e9:.1f} GB — starting download...")
colab_files.download(zip_path)
# (The unzipped bundle also remains in Drive under my_voice_clone/finetuned/)